In [5]:
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import time

# Update this path to wherever your 8GB parquet file actually lives on your local machine
GRID_FILE_PATH = "../data/hades_processed_grid.parquet" 

# Constants
M_J = 1.898e27
R_J = 69911000

In [14]:
# This reads ONLY the metadata, not the 8GB of data
parquet_file = pq.ParquetFile(GRID_FILE_PATH)

print(f"Total rows in file: {parquet_file.metadata.num_rows:,}")
print(f"Total row groups (chunks): {parquet_file.metadata.num_row_groups}")
print(f"Number of columns: {parquet_file.metadata.num_columns}")

# Let's peek at just the column names
print("\nFirst 15 columns:")
print(parquet_file.schema.names[:15])

ArrowInvalid: Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.

In [ ]:
start_time = time.time()

# The columns we care about for the ML model
needed_columns = [
    'mass', 'T_int', 'T_irr', 'Met', 'core', 'f_sed', 'kzz', 
    'Req', 'S_physical', 'dsdt'
]

# We must use the raw units that exist IN the file for the filters
mass_threshold_kg = 5 * M_J

# Predicate pushdown: filter the data BEFORE it enters RAM
filters = [
    ('T_int', '<', 500),
    ('mass', '<=', mass_threshold_kg)
]

print("Loading filtered slice...")
df_slice = pd.read_parquet(
    GRID_FILE_PATH, 
    engine='pyarrow',
    columns=needed_columns,
    filters=filters
)

print(f"Loaded {len(df_slice):,} rows in {time.time() - start_time:.2f} seconds!")
display(df_slice.head())

In [ ]:
# Convert to Jupiter units
df_slice['mass_Mj'] = df_slice['mass'] / M_J
df_slice['Req_Rj'] = df_slice['Req'] / R_J

# Target variable for the ODE
df_slice['abs_log_dsdt'] = np.log10(np.abs(df_slice['dsdt']))

# Drop any NaNs
df_slice = df_slice.dropna().reset_index(drop=True)

print("Data preprocessed successfully. Final shape:", df_slice.shape)
display(df_slice[['mass_Mj', 'T_int', 'abs_log_dsdt']].head())